In [8]:
input_path = "../../data/input/"
output_path = "../../data/output/json/"

In [ ]:
import os
import json

def list_keys(d, lvl=0):
    if not isinstance(d, dict):
        return
    key_list = list(d.keys())
    key_list.sort()
    for key in key_list:
        print(' ' * lvl * 4 + key)
        list_keys(d[key], lvl + 1)

filepath = "../../Test_01_rules_input_20260429_192517.json"
with open(filepath, "r", encoding="utf-8") as f:
    data_dict = json.load(f)
list_keys(data_dict)

In [7]:
import os
import shutil
import time
import trimesh
import requests
import json
import urllib
import numpy as np
import matplotlib.pyplot as plt

# ======== Load credentials ========
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_group = "APIClient"
    user_id = creds['user_id']
    print("Successfully loaded credentials from creds.json")
else:
    raise RuntimeError("Credentials not found")

# ======== API access ========
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/input/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # Must specify postfix, i.e., file extension
                        headers={"X-ZH-TOKEN": zh_token}) # Get signed upload URL
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # Returns a single string JSON "string", can also use json.loads(resp.text)

    resp = requests.put(upload_url, data) # No auth header is needed for uploading to the cloud storage service

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
        "Content-Type": "application/json",
        "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)

    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")
    print(f"API finished in {time.time() - start_time}s")

    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_data(urn):
    return requests.get(
        file_server_url + f"/file/download?" + \
            urllib.parse.urlencode({"urn": urn}),
        headers={"X-ZH-TOKEN": zh_token}
    ).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(
        file_server_url + f"/file/download?" + \
            urllib.parse.urlencode({"urn": mesh_file_json['data']}),
        headers={"X-ZH-TOKEN": zh_token}
    )
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])


# ======== Analyses ========
def upper_teeth(filepath):
    json_call = {
        "spec_group": "mesh-processing",
        "spec_name": "oral-denoise-prod",
        "spec_version": "1.0-snapshot",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "mesh": {"type": "drc", "data": upload_file(filepath)},
            "jaw_type": "Upper"
        },
        "output_config": {
            "teeth_comp": {"type": "ply"}
        }
    }
    print('Performing upper jaw analysis...')
    return run_job_and_get_results(json_call, 300)

def lower_teeth(filepath):
    json_call = {
        "spec_group": "mesh-processing",
        "spec_name": "oral-denoise-prod",
        "spec_version": "1.0-snapshot",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "mesh": {"type": "ply", "data": upload_file(filepath)},
            "jaw_type": "Lower"
        },
        "output_config": {
            "teeth_comp": {"type": "ply"}
        }
    }
    print('Performing lower jaw analysis...')
    return run_job_and_get_results(json_call, 300)

def auto_arrange(result_upper_jaw, result_lower_jaw):
    json_call = {
        "spec_group": "mesh-processing",
        "spec_name": "auto-arrange", 
        "spec_version": "1.0-snapshot",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "upper_teeth_dict": result_upper_jaw["teeth_comp"],
            "upper_axis_matrix_dict": result_upper_jaw["axis"],
            "lower_teeth_dict": result_lower_jaw["teeth_comp"],
            "lower_axis_matrix_dict": result_lower_jaw["axis"],
        }
    }
    print('Performing auto arrangement...')
    return run_job_and_get_results(json_call, 500)

def case_complexity(result_upper_jaw, result_lower_jaw, result_arrangement):
    json_call = {
        "spec_group": "mesh-processing",
        "spec_name": "case-complexity-analysis", 
        "spec_version": "1.0-snapshot",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "teeth_dict": {**result_upper_jaw["teeth_comp"], **result_lower_jaw["teeth_comp"]},
            "axis_dict": {**result_upper_jaw["axis"], **result_lower_jaw["axis"]},
            "transformation_dict": result_arrangement["result"]["transformation_dict"],
            "landmarks_dict": {**result_upper_jaw["landmarks"], **result_lower_jaw["landmarks"]},
            "target_out_form": "" # No function for now, pass empty string
        }
    }
    print('Performing case complexity analysis...')
    return run_job_and_get_results(json_call, 500)

def ceph(filepath):
    json_call = {
        "spec_version": "1.0-snapshot",
        "spec_name": "ceph-analysis",
        "spec_group": "ceph",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "image": upload_file(filepath)
        }
    }
    print('Performing cephalometric image analysis...')
    return run_job_and_get_results(json_call, 180)

def pano(filepath):
    json_call = {
        "spec_version": "1.0-snapshot",
        "spec_name": "pano-analysis",
        "spec_group": "pano",
        "user_group": user_group,
        "user_id": user_id,
        "input_data": {
            "image": upload_file(filepath)
        }
    }
    print('Performing panoramic radiograph analysis...')
    return run_job_and_get_results(json_call, 180)

Successfully loaded credentials from creds.json


In [9]:
res_upper = upper_teeth(os.path.join(input_path, "upper_jaw_scan.drc"))
with open(os.path.join(output_path, "upper_jaw.json"), 'w') as f:
    json.dump(res_upper, f, indent=2, ensure_ascii=False)

Performing upper jaw analysis...
workflow id is wf_1784837325-a493ce11-234a-4b90-8493-e5be57973ef9
API finished in 87.2144923210144s


In [10]:
res_lower = lower_teeth(os.path.join(input_path, "lower_jaw_scan.ply"))
with open(os.path.join(output_path, "lower_jaw.json"), 'w') as f:
    json.dump(res_lower, f, indent=2, ensure_ascii=False)

Performing lower jaw analysis...
workflow id is wf_1784837467-f6dc1e5a-fa6d-4c98-b056-38aa1e16e365
API finished in 92.29724454879761s


In [11]:
res_arr = auto_arrange(res_upper, res_lower)
with open(os.path.join(output_path, "auto_arrange.json"), 'w') as f:
    json.dump(res_arr, f, indent=2, ensure_ascii=False)

Performing auto arrangement...
workflow id is sa_service_1784837563-ca4d41cf-27ad-495a-a49d-6a6957cb9287
API finished in 12.480478286743164s


In [12]:
res_complex = case_complexity(res_upper, res_lower, res_arr)
with open(os.path.join(output_path, "case_complexity.json"), 'w') as f:
    json.dump(res_complex, f, indent=2, ensure_ascii=False)

Performing case complexity analysis...
workflow id is sa_service_1784837577-ea572f4e-bf12-4273-8c76-5302bcda2a6e
API finished in 17.5820631980896s


In [ ]:
res_ceph = ceph(os.path.join(input_path, "ceph.jpg"))
with open(os.path.join(output_path, "ceph.json"), 'w') as f:
    json.dump(res_ceph, f, indent=2, ensure_ascii=False)

In [14]:
res_pano = pano(os.path.join(input_path, "pano.jpg"))
with open(os.path.join(output_path, "pano.json"), 'w') as f:
    json.dump(res_pano, f, indent=2, ensure_ascii=True)

Performing panoramic radiograph analysis...
workflow id is sa_service_1784837790-06d8742b-313b-4e99-aecc-96d948fd6942
API finished in 2.521864891052246s


In [ ]:
import json
import os

# ======== Main ========
input_path = "../../data/input/"
output_path = "../../data/output/json/"

res_upper = json.load(open(os.path.join(output_path, 'upper_jaw.json'), 'r'))
res_lower = json.load(open(os.path.join(output_path, 'lower_jaw.json'), 'r'))
res_arr = json.load(open(os.path.join(output_path, 'auto_arrange.json'), 'r'))
res_ceph = json.load(open(os.path.join(output_path, 'ceph.json'), 'r'))
res_pano = json.load(open(os.path.join(output_path, 'pano.json'), 'r'))
res_complex = json.load

arranged = {
    'analysis' : {
        'auto_arrange' : res_arr,
        'case_complexity' : res_complex,
        'ceph' : res_ceph,
        'pano' : res_pano,
        'teeth_segmentation' : {
            'result' : {
                'lower' : res_lower,
                'upper' : res_upper
            }
        }
    }
}

output_file_path = os.path.join(output_path, "rule_eng_input.json")
with open(output_file_path, 'w') as f:
    json.dump(arranged, f, indent=2, ensure_ascii=True)

print("All analyses completed. Output JSON file in \"" 
      + os.path.abspath(output_file_path) + "\".")

NameError: name 'res_complex' is not defined

In [22]:
with open(output_file_path, 'r', encoding='utf-8') as f:
    data_dict = json.load(f)
list_keys(data_dict)

analysis
    auto_arrange
        result
            form
            transformation_dict
                11
                12
                13
                14
                15
                16
                17
                21
                22
                23
                24
                25
                26
                27
                31
                32
                33
                34
                35
                36
                37
                42
                43
                44
                45
                46
                47
    case_complexity
        result
            anterior_intrusion_per_tooth
                11
                12
                13
                21
                22
                23
                31
                32
                33
                42
                43
            ipr_per_contact
            left_class_2_discrepancy
            left_class_3_discrepancy
          